In [1]:
#Définition du Schéma de Sortie (Pydantic)
from pydantic import BaseModel, Field

class MetaPrompts(BaseModel):
    debutant: str = Field(description="Prompt vulgarisé et pédagogique")
    expert: str = Field(description="Prompt technique avec jargon métier")
    manager: str = Field(description="Prompt orienté valeur ajoutée et synthèse")

In [2]:
#Création de la Chaîne de Meta-Prompting
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Configuration du parser et du modèle
parser = JsonOutputParser(pydantic_object=MetaPrompts)
OLLAMA_HOST =   "http://host.docker.internal:11434"
llm = ChatOllama(model="gemma3:4b", base_url=OLLAMA_HOST, temperature=0.7) # Ou tout modèle supportant cette fonction

# 2. Gabarit de Meta-Prompt
META_PROMPT_SYSTEM = """
Tu es un ingénieur de prompt expert. Ton rôle est de multiplier une consigne source 
en trois variantes adaptées à des audiences différentes.
{format_instructions}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", META_PROMPT_SYSTEM),
    ("human", "Génère les variantes pour cette consigne : {consigne_source}")
])

# 3. Assemblage de la chaîne LCEL (Pipeline) 
meta_chain = (
    prompt_template.partial(format_instructions=parser.get_format_instructions())
    | llm 
    | parser
)

In [3]:
# Exécution du Workflow
consigne = "Expliquer pourquoi le VPN GlobalProtect peut échouer à cause du port 443"

# Lancement de la génération [13]
variantes = meta_chain.invoke({"consigne_source": consigne})

# Affichage des résultats
print(f"--- VERSION DÉBUTANT ---\n{variantes['debutant']}\n")
print(f"--- VERSION EXPERT ---\n{variantes['expert']}\n")
print(f"--- VERSION MANAGER ---\n{variantes['manager']}")

--- VERSION DÉBUTANT ---
Le GlobalProtect ne marche pas ? C'est souvent parce qu'il essaie de communiquer via internet en utilisant le protocole HTTPS (comme quand on visite un site web). Le port 443 est utilisé par HTTPS. Si quelque chose bloque cette communication, GlobalProtect ne peut pas se connecter. C'est comme si on essayait de parler à quelqu'un mais que le téléphone était coupé.

--- VERSION EXPERT ---
L'échec de GlobalProtect sur le port 443 est fréquemment lié à des problèmes de configuration TLS/SSL, des restrictions de pare-feu ou des certificats non valides. Les causes peuvent inclure des erreurs de négociation TLS, des certificats expirés ou non installés, des configurations incorrectes de DNS, ou des interférences provenant d'autres applications utilisant le port 443. Une analyse approfondie des journaux d'événements et des outils de diagnostic réseau est nécessaire pour identifier la cause racine.

--- VERSION MANAGER ---
Le problème de connectivité GlobalProtect via 